In [1]:
from google.colab import files

uploaded = files.upload()

Saving base_datos_01.csv to base_datos_01.csv


In [2]:
import pandas as pd

df = pd.read_csv("base_datos_01.csv")

df.head()

,pais_escucha,nacionalidad_artista,streams_total,artistas_distintos,canciones_distintas,total_pais,peso_relativo
0,Argentina,Chile,12245430,8,4,285362921,4.29
1,Argentina,Argentina,117428622,53,50,285362921,41.15
2,Argentina,Colombia,32150701,9,9,285362921,11.27
3,Argentina,México,7099697,2,2,285362921,2.49
4,Argentina,Puerto Rico,62091505,25,28,285362921,21.76


In [3]:
print(df.columns)
print(df["pais_escucha"].unique())
print(df["nacionalidad_artista"].unique())

Index(['pais_escucha', 'nacionalidad_artista', 'streams_total',
       'artistas_distintos', 'canciones_distintas', 'total_pais',
       'peso_relativo'],
      dtype='object')
['Argentina' 'Chile' 'Colombia' 'México']
['Chile' 'Argentina' 'Colombia' 'México' 'Puerto Rico' 'Otros']


In [4]:
import pandas as pd
from IPython.display import HTML, display

# =========================
# 1. Cargar base
# =========================

df = pd.read_csv("base_datos_01.csv")

# Estandarizar nombre de México si viniera sin tilde
df["pais_escucha"] = df["pais_escucha"].replace({"Mexico": "México"})

# Orden visual de países y categorías
orden_paises = ["Chile", "Argentina", "Colombia", "México"]
categorias = ["Chile", "Argentina", "Colombia", "México", "Puerto Rico", "Otros"]

# Colores usados en la visualización
colores = {
    "Chile": "#20bf55",
    "Argentina": "#8c5bf3",
    "Colombia": "#ff7a1a",
    "México": "#f4d118",
    "Puerto Rico": "#ff5252",
    "Otros": "#9aa9bf"
}

# =========================
# 2. Preparar datos
# =========================

# Convertir la base en una estructura más fácil de usar
valores = {
    (fila["pais_escucha"], fila["nacionalidad_artista"]): float(fila["peso_relativo"])
    for _, fila in df.iterrows()
}

datos = []

for pais in orden_paises:
    fila_pais = []
    for categoria in categorias:
        valor = valores.get((pais, categoria), 0)
        fila_pais.append({
            "categoria": categoria,
            "valor": valor,
            "color": colores[categoria]
        })
    datos.append({
        "pais": pais,
        "segmentos": fila_pais
    })

# =========================
# 3. Función para crear cada barra
# =========================

def crear_barra(pais, segmentos):
    html = f"""
    <div class="row">
      <div class="label">{pais}</div>
      <div class="bar-wrap">
    """

    acumulado = 0

    # Etiquetas pequeñas que van arriba de la barra
    for seg in segmentos:
        valor = seg["valor"]
        color = seg["color"]

        if valor > 0 and valor < 7:
            centro = acumulado + (valor / 2)

            # Ajustes manuales para que las etiquetas pequeñas no choquen tanto
            top = "-2px"
            left = centro

            if pais == "México" and valor <= 0.6:
                top = "-14px"
                left = max(1.8, centro)

            html += f"""
            <div class="leader" style="left:calc({centro}% - 1px); top:12px; height:14px; background:{color};"></div>
            <div class="smalltag" style="left:{left}%; top:{top}; border:1px solid {color};">{valor:.1f}%</div>
            """

        acumulado += valor

    html += '<div class="bar">'

    # Segmentos dentro de la barra
    for seg in segmentos:
        categoria = seg["categoria"]
        valor = seg["valor"]
        color = seg["color"]

        if valor > 0:
            etiqueta = f"{valor:.1f}%" if valor >= 7 else ""
            html += f"""
            <div class="seg" style="width:{valor}%;background:{color};">{etiqueta}</div>
            """

    html += """
        </div>
      </div>
    </div>
    """

    return html

# =========================
# 4. Crear HTML completo
# =========================

html = """
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>¿De dónde viene la música que escuchamos?</title>

<style>
  * {
    box-sizing: border-box;
  }

  html, body {
    margin: 0;
    padding: 0;
    background: #07152a;
    font-family: Arial, Helvetica, sans-serif;
    color: #fff;
  }

  .card {
    width: 100%;
    margin: 0;
    background: #07152a;
    color: #fff;
    border-radius: 26px;
    padding: 18px 22px 18px;
  }

  .title {
    margin: 0 0 10px;
    font-size: 32px;
    line-height: 1.05;
    font-weight: 800;
  }

  .subtitle {
    font-size: 17px;
    line-height: 1.4;
    color: #dbe6f4;
    max-width: 100%;
    margin-bottom: 14px;
  }

  .row {
    display: grid;
    grid-template-columns: 130px 1fr;
    align-items: center;
    gap: 14px;
    margin: 18px 0;
  }

  .label {
    font-size: 17px;
    font-weight: 700;
  }

  .bar-wrap {
    position: relative;
    padding-top: 26px;
  }

  .bar {
    display: flex;
    width: 100%;
    height: 40px;
    border-radius: 999px;
    overflow: hidden;
    background: #13223a;
  }

  .seg {
    height: 100%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 12px;
    font-weight: 700;
    color: #111;
  }

  .smalltag {
    position: absolute;
    transform: translateX(-50%);
    font-size: 11px;
    font-weight: 700;
    color: #fff;
    padding: 2px 5px;
    border-radius: 7px;
    background: #07152a;
    white-space: nowrap;
    line-height: 1.1;
  }

  .leader {
    position: absolute;
    width: 2px;
  }

  .legend {
    display: flex;
    gap: 18px;
    flex-wrap: wrap;
    align-items: center;
    margin-top: 18px;
  }

  .legend-item {
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 14px;
  }

  .dot {
    width: 16px;
    height: 16px;
    border-radius: 6px;
    display: inline-block;
  }

  .footnote {
    margin-top: 14px;
    font-size: 13px;
    color: #b9c7da;
    line-height: 1.35;
  }

  @media (max-width: 900px) {
    .card {
      padding: 16px;
    }

    .title {
      font-size: 27px;
    }

    .subtitle {
      font-size: 15px;
    }

    .row {
      grid-template-columns: 110px 1fr;
      gap: 10px;
      margin: 14px 0;
    }

    .label {
      font-size: 16px;
    }

    .bar {
      height: 34px;
    }

    .seg {
      font-size: 11px;
    }
  }

  @media (max-width: 640px) {
    .title {
      font-size: 22px;
    }

    .subtitle {
      font-size: 14px;
    }

    .row {
      grid-template-columns: 1fr;
      gap: 6px;
      margin: 14px 0 18px;
    }

    .label {
      font-size: 15px;
    }

    .bar-wrap {
      padding-top: 26px;
    }

    .bar {
      height: 30px;
    }

    .seg {
      font-size: 10px;
    }

    .smalltag {
      font-size: 9px;
      padding: 2px 4px;
    }

    .legend {
      gap: 10px;
    }

    .legend-item {
      font-size: 12px;
    }

    .dot {
      width: 13px;
      height: 13px;
    }
  }
</style>
</head>

<body>
<div class="card">
  <div class="title">¿De dónde viene la música que escuchamos?</div>
  <div class="subtitle">
    Peso relativo de las nacionalidades de artistas dentro de los charts analizados.
    Los streams fueron fraccionados entre artistas cuando una canción tiene colaboraciones.
  </div>
"""

# Agregar barras por país
for item in datos:
    html += crear_barra(item["pais"], item["segmentos"])

# Agregar leyenda
html += """
  <div class="legend">
"""

for categoria in categorias:
    html += f"""
    <div class="legend-item">
      <span class="dot" style="background:{colores[categoria]}"></span>{categoria}
    </div>
    """

html += """
  </div>

  <div class="footnote">
    Categorías: Chile, Argentina, Colombia, México, Puerto Rico y Otros.
    Base: Spotify Charts, 4 países, 6 meses.
  </div>
</div>
</body>
</html>
"""

# =========================
# 5. Guardar archivo HTML
# =========================

with open("grafico_origen_musica.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Archivo generado: grafico_origen_musica.html")

Archivo generado: grafico_origen_musica.html


In [7]:
from google.colab import files

files.download("grafico_origen_musica.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from IPython.display import IFrame

IFrame("grafico_origen_musica.html", width="100%", height=560)